In [0]:
storage_account_name = "adlsgen1111"
container_name = "gold"
access_key = "PphoiAnr9BY7PAveVPLg/m+msI8DzTx4RJA6PDZJVg+iZ0ZivJR/4YLuS+/pKqOsKxOS/4WMx5k2+AStr72IYA==" 
try:
    dbutils.fs.unmount(f"/mnt/{container_name}")
    print("Unmounted successfully.")
except:
    print("Nothing to unmount.")
mount_point = f"/mnt/{container_name}"
source_url = f"wasbs://{container_name}@{storage_account_name}.blob.core.windows.net"

try:
    dbutils.fs.mount(
        source = source_url,
        mount_point = mount_point,
        extra_configs = {f"fs.azure.account.key.{storage_account_name}.blob.core.windows.net": access_key}
    )
    print(f"Success! Mounted /mnt/{container_name} to Azure Storage.")
except Exception as e:
    print(" Mount failed or already exists:", e)


/mnt/gold has been unmounted.
Unmounted successfully.
Success! Mounted /mnt/gold to Azure Storage.


In [0]:
from pyspark.sql.functions import col, lit, when

df_input = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/mnt/silver/ml_out_smart_city/energy_forecast.csv")

df_input = df_input.withColumnRenamed("Scored Labels", "scored_labels")

df_forecast = df_input.withColumn("predicted_consumption", col("energy_consumption") * 1.05) \
                      .withColumn("prediction_error", lit(78.2))

df_forecast.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/mnt/gold/energy_forecast")

print(" Step 1 Done: Table 'energy_forecast' created and Saved in Gold!")
display(df_forecast.limit(5))

 Step 1 Done: Table 'energy_forecast' created and Saved in Gold!


reading_time,region_id,energy_consumption,peak_hours_flag,month,weekday,feels_like_raw,humidity_raw,weather_index,scored_labels,predicted_consumption,prediction_error
2025-12-06 12:00:00,r1,95.44565467551894,0,12,7,18.108598726114653,66,null,264.15355159216114,100.21793740929489,78.2
2025-12-02 06:00:00,r4,305.05453141473254,0,12,3,13.7,93,37.489999999999995,254.75274746174793,320.30725798546916,78.2
2025-11-22 04:00:00,r2,318.7153993362996,0,11,7,18.108598726114653,66,null,245.43035205516907,334.65116930311456,78.2
2025-12-23 07:00:00,r4,391.0482600549623,0,12,3,18.108598726114653,66,null,248.9249456284177,410.6006730577105,78.2
2025-11-17 16:00:00,r3,87.71309066353044,0,11,2,18.108598726114653,66,null,250.10826734954102,92.09874519670696,78.2


In [0]:
df_forecast = spark.read.format("delta").load("/mnt/gold/energy_forecast")

PEAK_THRESHOLD = 2000 
CRITICAL_ZONES = [101, 105, 110]

df_optimization = df_forecast.withColumn(
    "optimization_action",
    when(
        (col("predicted_consumption") > PEAK_THRESHOLD) & (col("region_id").isin(CRITICAL_ZONES)),
        "CRITICAL: Activate Backup Generators"
    )
    .when(
        (col("predicted_consumption") > PEAK_THRESHOLD),
        
        "Peak Shifting: Reduce HVAC"
    )
    .when(
        (col("predicted_consumption") > (PEAK_THRESHOLD * 0.8)),
        "Load Balancing: Transfer load"
    )
    .otherwise(" System Stable")
)

df_optimization.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("/mnt/gold/optimization_recommendations")

print("Step 2 Done: Optimization Recommendations created!")
display(
    df_optimization.select(
        "region_id",
        "predicted_consumption",
        "optimization_action"
    ).limit(5)
)

Step 2 Done: Optimization Recommendations created!


region_id,predicted_consumption,optimization_action
r1,100.21793740929489,System Stable
r4,320.30725798546916,System Stable
r2,334.65116930311456,System Stable
r4,410.6006730577105,System Stable
r3,92.09874519670696,System Stable


In [0]:
display(dbutils.fs.mounts())

mountPoint,source,encryptionType
/databricks-datasets,databricks-datasets,
/mnt/gold,wasbs://gold@adlsgen1111.blob.core.windows.net,
/Volumes,UnityCatalogVolumes,
/mnt/silver,wasbs://silver@adlsgen1111.blob.core.windows.net/,
/databricks/mlflow-tracking,databricks/mlflow-tracking,
/databricks-results,databricks-results,
/mnt/raw,wasbs://raw@adlsgen1111.blob.core.windows.net/,
/databricks/mlflow-registry,databricks/mlflow-registry,
/mnt/bronze,wasbs://bronze@adlsgen1111.blob.core.windows.net/,
/Volume,DbfsReserved,


In [0]:
display(dbutils.fs.ls("/mnt/gold/"))

path,name,size,modificationTime
dbfs:/mnt/gold/energy_forecast/,energy_forecast/,0,0
dbfs:/mnt/gold/optimization_recommendations/,optimization_recommendations/,0,0
dbfs:/mnt/gold/risk_zones/,risk_zones/,0,0


In [0]:
from pyspark.sql.functions import col, when

df_forecast = spark.read.format("delta").load("/mnt/gold/energy_forecast")

peak_threshold = 2000 

df_risk = df_forecast.withColumn(
    "risk_label",
    when(col("predicted_consumption") > peak_threshold, "High Risk")
    .when(col("predicted_consumption") > (peak_threshold * 0.8), "Medium Risk")
    .otherwise("Low Risk")
)
df_risk.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("/mnt/gold/risk_zones")

print(" Gold Table Created: risk_zones")
display(df_risk.limit(5))

 Gold Table Created: risk_zones


reading_time,region_id,energy_consumption,peak_hours_flag,month,weekday,feels_like_raw,humidity_raw,weather_index,scored_labels,predicted_consumption,prediction_error,risk_label
2025-12-06 12:00:00,r1,95.44565467551894,0,12,7,18.108598726114653,66,null,264.15355159216114,100.21793740929489,78.2,Low Risk
2025-12-02 06:00:00,r4,305.05453141473254,0,12,3,13.7,93,37.489999999999995,254.75274746174793,320.30725798546916,78.2,Low Risk
2025-11-22 04:00:00,r2,318.7153993362996,0,11,7,18.108598726114653,66,null,245.43035205516907,334.65116930311456,78.2,Low Risk
2025-12-23 07:00:00,r4,391.0482600549623,0,12,3,18.108598726114653,66,null,248.9249456284177,410.6006730577105,78.2,Low Risk
2025-11-17 16:00:00,r3,87.71309066353044,0,11,2,18.108598726114653,66,null,250.10826734954102,92.09874519670696,78.2,Low Risk
